# DeepTMHMM pipeline for Google Colab

Predicts transmembrane topology (α-helical TM, β-barrel, signal peptide, globular) with **DeepTMHMM**, run through the official BioLib Python client (`pybiolib`). Jobs execute on BioLib's cloud, so no GPU is needed in Colab.

**Steps**
1. Install dependencies
2. (Optional) sign in to BioLib for higher quotas
3. Provide sequences — download straight from UniProt (accessions, search query or proteome), upload a FASTA, or paste
4. Run DeepTMHMM in batches (with retries)
5. Parse results into a table
6. Plot topologies
7. Export & download

> Cite: Hallgren *et al.* (2022) *DeepTMHMM predicts alpha and beta transmembrane proteins using deep neural networks.* bioRxiv doi:10.1101/2022.04.08.487609. Commercial use requires a licence (licensing@biolib.com).

## 1. Install

In [ ]:
!pip install -q pybiolib biopython pandas matplotlib requests
import biolib, Bio
print("pybiolib", getattr(biolib, "__version__", "?"), "| biopython", Bio.__version__)

## 2. (Optional) BioLib sign-in
Anonymous runs work, but signing in gives you a job history and more generous limits. Create a token at biolib.com → Account → API tokens, then add it to Colab **Secrets** (key icon in the left sidebar) as `BIOLIB_TOKEN`. Skip this cell to run anonymously.

In [ ]:
import os
try:
    from google.colab import userdata
    token = userdata.get("BIOLIB_TOKEN")
    if token:
        os.environ["BIOLIB_TOKEN"] = token
        print("BioLib token loaded from Colab Secrets.")
except Exception:
    print("No BIOLIB_TOKEN secret found — running anonymously.")

## 3. Configure & provide sequences
Choose **one** input mode (`uniprot`, `upload` or `paste`). Settings:
- `BATCH_SIZE` — sequences per BioLib job (smaller batches are more robust; 100–500 is sensible)
- `MAX_RETRIES` — re-submissions per failed batch

In [ ]:
#@title Settings { display-mode: "form" }
INPUT_MODE = "uniprot"  #@param ["uniprot", "upload", "paste"]
BATCH_SIZE = 200      #@param {type:"integer"}
MAX_RETRIES = 3       #@param {type:"integer"}
RUN_NAME = "deeptmhmm_run"  #@param {type:"string"}

# Used when INPUT_MODE == "paste"
PASTED_FASTA = """
>sp|P02945|BACR_HALSA Bacteriorhodopsin
MLELLPTAVEGVSQAQITGRPEWIWLALGTALMGLGTLYFLVKGMGVSDPDAKKFYAITTLVPAIAFTMYLSMLLGYGLTMVPFGGEQNPIYWARYADWLFTTPLLLLDLALLVDADQGTILALVGADGIMIGTGLVGALTKVYSYRFVWWAISTAAMLYILYVLFFGFTSKAESMRPEVASTFKVLRNVTVVLWSAYPVVWLIGSEGAGIVPLNIETLLFMVLDVSAKVGFGLILLRSRAIFGEAEAPEPSAGDGAAATSD
"""

### 3a. Download sequences directly from UniProt
Used when `INPUT_MODE = "uniprot"`. Pick a **source**:

| `UNIPROT_SOURCE` | what to enter in `UNIPROT_INPUT` | example |
|---|---|---|
| `accessions` | accessions or entry names, separated by commas/spaces/newlines | `P02945, P0A917, OMPX_ECOLI` |
| `query` | any UniProt search query ([syntax](https://www.uniprot.org/help/query-fields)) | `organism_id:83333 AND keyword:KW-0812 AND reviewed:true` |
| `proteome` | a proteome ID | `UP000000625` (E. coli K-12) |

`MAX_SEQUENCES` caps the download (0 = no cap). `REVIEWED_ONLY` adds `reviewed:true` (Swiss-Prot) to query/proteome searches. `INCLUDE_ISOFORMS` also downloads isoform sequences. The downloaded FASTA and a metadata table are saved to the run folder.

In [ ]:
#@title UniProt download { display-mode: "form" }
UNIPROT_SOURCE = "accessions"  #@param ["accessions", "query", "proteome"]
UNIPROT_INPUT = "P02945, P0A917, P69905"  #@param {type:"string"}
MAX_SEQUENCES = 500     #@param {type:"integer"}
REVIEWED_ONLY = False   #@param {type:"boolean"}
INCLUDE_ISOFORMS = False  #@param {type:"boolean"}

In [ ]:
import io, re, time, requests
import pandas as pd
from pathlib import Path
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

WORKDIR = Path("/content") / RUN_NAME
WORKDIR.mkdir(parents=True, exist_ok=True)

UNIPROT = "https://rest.uniprot.org/uniprotkb"
session = requests.Session()
session.headers["User-Agent"] = "DeepTMHMM-Colab-pipeline"

def _get(url, params=None, tries=4):
    for t in range(tries):
        r = session.get(url, params=params, timeout=60)
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** t * 2); continue
        r.raise_for_status()
        return r
    r.raise_for_status()

def _next_link(r):
    m = re.search(r'<([^>]+)>;\s*rel="next"', r.headers.get("Link", ""))
    return m.group(1) if m else None

def uniprot_search(query, fmt="fasta", fields=None, limit=0, page_size=500):
    """Paginated UniProtKB search; returns concatenated text."""
    params = {"query": query, "format": fmt, "size": page_size}
    if fields: params["fields"] = fields
    if fmt == "fasta" and INCLUDE_ISOFORMS: params["includeIsoform"] = "true"
    url, chunks, n = f"{UNIPROT}/search", [], 0
    while url:
        r = _get(url, params)
        params = None  # the next-link already contains the params
        if n == 0:
            print(f"  UniProt reports {r.headers.get('X-Total-Results', '?')} matching entries")
        text = r.text
        if fmt == "tsv" and chunks:
            text = text.split("\n", 1)[1] if "\n" in text else ""
        chunks.append(text)
        n += text.count(">") if fmt == "fasta" else max(text.count("\n") - (n == 0), 0)
        print(f"  downloaded {n}…", end="\r")
        if limit and n >= limit: break
        url = _next_link(r)
    print()
    return "".join(chunks)

def build_query():
    raw = UNIPROT_INPUT.strip()
    if UNIPROT_SOURCE == "accessions":
        ids = [i for i in re.split(r"[,;\s]+", raw) if i]
        if not ids: raise ValueError("No accessions given.")
        return ids, None
    if UNIPROT_SOURCE == "proteome":
        q = f"proteome:{raw}"
    else:
        q = raw
    if REVIEWED_ONLY and "reviewed:" not in q:
        q = f"({q}) AND reviewed:true"
    return None, q

META_FIELDS = "accession,id,protein_name,gene_names,organism_name,length,reviewed,cc_subcellular_location"

def _acc_query(ids):
    # accession:X matches primary accessions; id:X matches entry names like OMPX_ECOLI
    return " OR ".join(f"id:{x}" if "_" in x else f"accession:{x}" for x in ids)

def fetch_uniprot():
    ids, query = build_query()
    if ids:
        queries = [_acc_query(ids[i:i + 100]) for i in range(0, len(ids), 100)]  # keep URLs short
        limit = 0
    else:
        print("Query:", query)
        queries, limit = [query], MAX_SEQUENCES

    records, metas = [], []
    for q in queries:
        records += list(SeqIO.parse(io.StringIO(uniprot_search(q, limit=limit)), "fasta"))
        try:
            tsv = uniprot_search(q, fmt="tsv", fields=META_FIELDS, limit=limit)
            metas.append(pd.read_csv(io.StringIO(tsv), sep="\t"))
        except Exception as e:
            print("  (metadata download skipped:", e, ")")

    if ids:
        got = set()
        for r in records:
            p = r.id.split("|")
            got.update(p[1:3] if len(p) >= 3 else [r.id])
        missing = [x for x in ids if x not in got]
        if missing:
            print("  ! not found in UniProtKB:", ", ".join(missing))
    if MAX_SEQUENCES and len(records) > MAX_SEQUENCES:
        print(f"  capping at MAX_SEQUENCES = {MAX_SEQUENCES}")
        records = records[:MAX_SEQUENCES]

    meta = pd.concat(metas, ignore_index=True).drop_duplicates("Entry") if metas else None
    if meta is not None:
        meta.to_csv(WORKDIR / "uniprot_metadata.tsv", sep="\t", index=False)
    SeqIO.write(records, WORKDIR / "uniprot_download.fasta", "fasta")
    print(f"Saved {len(records)} sequences → {WORKDIR / 'uniprot_download.fasta'}")
    return records, meta

def records_from_text(text):
    return list(SeqIO.parse(io.StringIO(text.strip()), "fasta"))

uniprot_meta = None
if INPUT_MODE == "uniprot":
    records, uniprot_meta = fetch_uniprot()
elif INPUT_MODE == "upload":
    from google.colab import files
    up = files.upload()
    records = []
    for name, data in up.items():
        records += records_from_text(data.decode())
elif INPUT_MODE == "paste":
    records = records_from_text(PASTED_FASTA)
else:
    raise ValueError("INPUT_MODE must be uniprot, upload or paste")

# --- clean & validate ---------------------------------------------------
VALID = set("ACDEFGHIKLMNPQRSTVWYXBZUO")
clean, id_map, acc_map, seen = [], {}, {}, set()
for i, rec in enumerate(records):
    seq = str(rec.seq).upper().replace("*", "").replace("-", "")
    bad = set(seq) - VALID
    if not seq or bad:
        print(f"  ! skipping {rec.id}: empty or invalid characters {sorted(bad)}")
        continue
    # UniProt headers look like sp|P02945|BACR_HALSA — use the accession as the ID
    parts = rec.id.split("|")
    base = parts[1] if len(parts) >= 3 and parts[0] in ("sp", "tr") else rec.id
    safe = re.sub(r"[^A-Za-z0-9_.-]", "_", base)[:60] or f"seq{i}"
    while safe in seen:
        safe += "_dup"
    seen.add(safe)
    id_map[safe] = rec.description
    clean.append(SeqRecord(Seq(seq), id=safe, description=""))

print(f"{len(clean)} valid sequences (of {len(records)} read)")
if not clean:
    raise SystemExit("No sequences to run.")
if uniprot_meta is not None:
    display(uniprot_meta.head(10))

## 4. Run DeepTMHMM on BioLib
Each batch is a separate BioLib job. Jobs queue on shared compute — expect a few minutes per batch. Completed batches are skipped if you re-run this cell, so you can resume after a disconnect.

In [ ]:
import os, time, shutil, traceback

deeptmhmm = biolib.load("DTU/DeepTMHMM")
batches = [clean[i:i + BATCH_SIZE] for i in range(0, len(clean), BATCH_SIZE)]
print(f"{len(batches)} batch(es) of up to {BATCH_SIZE}")

def run_batch(idx, recs):
    bdir = WORKDIR / f"batch_{idx:03d}"
    out = bdir / "biolib_results"
    if (out / "predicted_topologies.3line").exists():
        print(f"[batch {idx}] already done — skipping")
        return out
    bdir.mkdir(exist_ok=True)
    SeqIO.write(recs, bdir / "input.fasta", "fasta")

    for attempt in range(1, MAX_RETRIES + 1):
        cwd = os.getcwd()
        try:
            os.chdir(bdir)
            t0 = time.time()
            print(f"[batch {idx}] submitting {len(recs)} seqs (attempt {attempt})…")
            job = deeptmhmm.cli(args="--fasta input.fasta")
            # Newer pybiolib returns a Job object; save its files explicitly.
            if hasattr(job, "save_files"):
                job.save_files("biolib_results")
            if (out / "predicted_topologies.3line").exists():
                print(f"[batch {idx}] done in {time.time() - t0:.0f}s")
                return out
            print(f"[batch {idx}] job finished but no 3line output found")
            if hasattr(job, "get_stdout"):
                print(job.get_stdout().decode(errors="ignore")[-2000:])
        except Exception as e:
            print(f"[batch {idx}] error: {e}")
            traceback.print_exc(limit=1)
        finally:
            os.chdir(cwd)
        shutil.rmtree(out, ignore_errors=True)
        time.sleep(30 * attempt)
    print(f"[batch {idx}] FAILED after {MAX_RETRIES} attempts")
    return None

result_dirs = [d for d in (run_batch(i, b) for i, b in enumerate(batches)) if d]
print(f"\n{len(result_dirs)}/{len(batches)} batches succeeded")

## 5. Parse results
DeepTMHMM's `predicted_topologies.3line` has three lines per protein: `>ID | TYPE`, the sequence, and a per-residue label string:

| label | meaning |
|---|---|
| `S` | signal peptide |
| `I` | inside (cytoplasm) |
| `O` | outside (non-cytoplasmic) |
| `M` | α-helical TM segment |
| `B` | β-strand (β-barrel) |
| `P` | periplasm |

Predicted `TYPE` is one of `TM`, `SP+TM`, `SP`, `BETA`, `GLOB`.

In [ ]:
import itertools

LABELS = {"S": "signal", "I": "inside", "O": "outside", "M": "TM_helix", "B": "beta_strand", "P": "periplasm"}

def parse_3line(path):
    lines = [l.rstrip("\n") for l in open(path) if l.strip()]
    for h, seq, topo in zip(lines[0::3], lines[1::3], lines[2::3]):
        pid, _, ptype = h[1:].partition("|")
        yield pid.strip(), ptype.strip(), seq.strip(), topo.strip()

def segments(topo):
    pos, out = 1, []
    for lab, grp in itertools.groupby(topo):
        n = len(list(grp))
        out.append((lab, pos, pos + n - 1))
        pos += n
    return out

rows, seg_rows = [], []
for d in result_dirs:
    for pid, ptype, seq, topo in parse_3line(d / "predicted_topologies.3line"):
        segs = segments(topo)
        tm = [s for s in segs if s[0] == "M"]
        beta = [s for s in segs if s[0] == "B"]
        sp = next((s for s in segs if s[0] == "S"), None)
        rows.append({
            "id": pid,
            "description": id_map.get(pid, ""),
            "type": ptype,
            "length": len(seq),
            "n_TM_helices": len(tm),
            "n_beta_strands": len(beta),
            "signal_peptide": f"{sp[1]}-{sp[2]}" if sp else "",
            "N_terminus": "inside" if topo.lstrip("S")[:1] == "I" else ("outside" if topo.lstrip("S")[:1] in "OP" else ""),
            "TM_regions": ";".join(f"{a}-{b}" for _, a, b in tm),
            "topology": topo,
        })
        seg_rows += [{"id": pid, "region": LABELS.get(l, l), "start": a, "end": b} for l, a, b in segs]

df = pd.DataFrame(rows)
if INPUT_MODE == "uniprot" and uniprot_meta is not None and "Entry" in uniprot_meta:
    keep = [c for c in ["Entry", "Entry Name", "Protein names", "Gene Names", "Organism", "Subcellular location [CC]"] if c in uniprot_meta]
    df = df.merge(uniprot_meta[keep].rename(columns={"Entry": "id"}), on="id", how="left")
seg_df = pd.DataFrame(seg_rows)
print(df["type"].value_counts().to_string())
df.drop(columns="topology").head(20)

## 6. Plot topologies

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

#@markdown Number of proteins to draw (most TM helices first)
N_PLOT = 20  #@param {type:"integer"}

COLORS = {"S": "#e6a23c", "I": "#4c78a8", "O": "#9ecae9", "M": "#d62728", "B": "#2ca02c", "P": "#b279a2"}
sub = df.sort_values(["n_TM_helices", "n_beta_strands"], ascending=False).head(N_PLOT)

fig, ax = plt.subplots(figsize=(12, 0.45 * len(sub) + 1.2))
for y, (_, r) in enumerate(sub.iterrows()):
    for lab, a, b in segments(r["topology"]):
        h = 0.6 if lab in "MBS" else 0.25
        ax.barh(y, b - a + 1, left=a - 1, height=h, color=COLORS.get(lab, "grey"), linewidth=0)
ax.set_yticks(range(len(sub)))
ax.set_yticklabels([f"{r.id}  ({r.type})" for r in sub.itertuples()], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Residue")
ax.legend(handles=[Patch(color=c, label=LABELS[k]) for k, c in COLORS.items()],
          bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(WORKDIR / "topology_plot.png", dpi=200)
plt.show()

## 7. Export & download
Writes a summary CSV, a per-region CSV, merged raw DeepTMHMM outputs, and zips everything.

In [ ]:
summary_csv = WORKDIR / "deeptmhmm_summary.csv"
regions_csv = WORKDIR / "deeptmhmm_regions.csv"
df.to_csv(summary_csv, index=False)
seg_df.to_csv(regions_csv, index=False)

# merge raw outputs across batches
for fname in ["predicted_topologies.3line", "TMRs.gff3"]:
    with open(WORKDIR / f"merged_{fname}", "w") as fo:
        for d in result_dirs:
            p = d / fname
            if p.exists():
                fo.write(p.read_text().rstrip("\n") + "\n")

zip_path = shutil.make_archive(str(WORKDIR), "zip", WORKDIR)
print("Saved:", zip_path)

from google.colab import files
files.download(zip_path)

### Optional: save to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
dest = Path("/content/drive/MyDrive/DeepTMHMM") / RUN_NAME
shutil.copytree(WORKDIR, dest, dirs_exist_ok=True)
print("Copied to", dest)